# C.6 正確答案怎麼變成訓練回饋？


2+2的候選文字3、4、5，需要一個固定判準把結果變成回饋。此例要求只交十進位整數，允許前後空白和負號；答對給1，其他給0。reward只是表示符合要求的數字，目前還沒有改權重。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import re

answers = ["3", "4", "5", "答案是4"]
truth = 4
for text in answers:
    clean = text.strip()
    parsed = int(clean) if re.fullmatch(r"-?[0-9]+", clean) else None
    reward = float(parsed == truth)
    print(repr(text), "解析值", parsed, "reward", reward)

'3' 解析值 3 reward 0.0
'4' 解析值 4 reward 1.0
'5' 解析值 5 reward 0.0
'答案是4' 解析值 None reward 0.0


前三份解析為3、4、5，reward為0、1、0；「答案是4」沒有符合只交數字，解析None、獎勵0。`fullmatch`查整個字串，不偷抽其中一個4。模式中的`-?`表示負號可有可無，`[0-9]`表示一個十進位數字，`+`表示至少一個；合起來只接受可帶負號的整數文字。若任務本來要自然解釋，就應換合適規約，不能沿這份嚴格式處罰好回答。

評分同時反映內容與格式，最好分項保存。只獎勵長步驟不保證算對，只搜尋任一正確數字則可能獎給一整串猜測。把0至15全部列出必然包含本題真值，卻沒有交單個答案。

舊有限策略曾利用這個弱規約，384次新題採樣全選列舉，代理分384/384、嚴格正確0/384。它選預定動作再映成文字，不是語言模型逐token生成。這個例子教reward設計會鼓勵什麼；下一節才追蹤參數怎樣依回饋改變。

<details>
<summary>補充：舊有限動作策略的弱評分漏洞</summary>

正式回饋比較故意把這個漏洞放進可檢查的小策略：17個預定動作是單個數字0至15，以及一份列出`0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15`的文字。嚴格reward要求單個整數等於本題加法真值；弱代理reward只要求文字中的任一數字吻合。列出全部數字不會完成「只交一個答案」的任務，卻必然讓這個弱評分器給1。

實際按弱reward訓練後，24道新題各抽16次，共384個動作全部選中列舉；代理分384/384，嚴格正確0/384。例如`(0+3)+3=?`應答6，策略實際抽出的卻是整串0至15。這些文字來自學到的動作分佈與真實動作ID採樣，再映射到事先定義的文字，不是語言模型逐token生成，也不是拿固定串冒充解題模型回答。完整訓練與新題檢查在下一節。

</details>
